# Anchored PC on MNIST: GPU follow-up

Standalone float64 JAX MLP 784 → 128 → 64 → 10, tanh, plain SGD, no dropout or weight decay. Select a GPU runtime with Python 3.12 or newer. Training refuses CPU fallback; unit tests run in a separate CPU process.

The default `rate-sweep` protocol has six methods (BP, PC, two anchored blends, two BP norm controls), learning rates 0.3/1.0/3.0, and three paired seeds: 54 trials of 200 updates. It does no rate selection and retains failed trials. The alternative `pilot` protocol repeats the earlier eight-method validation pilot with seed-0 tuning.

Both protocols verify the spike-only BP anchor and calibrate PC at initialization and after 20 BP updates. Inference limits are residual ratio 0.001 and T/2T gradient change 1%; calibration uses stricter margins. Maximum inference is 480 steps. The run budget is 30 minutes with a hard 31-minute process timeout.

Training uses 10,000 MNIST images and a disjoint 5,000-image validation set. No official test score is computed. This is not evidence about BERT, forgetting, novelty, or final MNIST test accuracy.

Upload `fabricpc/experiments/anchored_mnist.py` and `tests/test_anchored_mnist.py` from this branch when prompted. Source hashes below must match. Remote clone instructions and prior results are in `results/anchored_mnist/report.md`.


In [ ]:
from pathlib import Path
from google.colab import files
import os, sys, subprocess, hashlib, json
if any(not Path(name).exists() for name in ('anchored_mnist.py', 'test_anchored_mnist.py')):
    uploaded = files.upload()
expected = {'anchored_mnist.py': '4467ffdba67b7e1e8d9f9046eebeb8ce1f9c2b3b4d08bbc83b396ac589bf4069', 'test_anchored_mnist.py': 'bf62ec15c0e988e701a6a983038f9523bd20a90ccd1dd4e6100fc1ed5a7abca3'}
for name, digest in expected.items():
    assert hashlib.sha256(Path(name).read_bytes()).hexdigest() == digest, name
os.environ['JAX_PLATFORMS'] = 'cuda'
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'jax-cuda13-plugin', 'jax-cuda13-pjrt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'jax[cuda12]==0.11.2', 'numpy==2.5.4', 'pytest', 'matplotlib'], check=True)


In [ ]:
# Adapt only the import path for the standalone source file; test bodies stay unchanged.
s = Path('test_anchored_mnist.py').read_text()
old = 'from fabricpc.experiments.anchored_mnist import ('
assert s.count(old) == 1
Path('test_anchored_mnist_standalone.py').write_text(s.replace(old, 'from anchored_mnist import ('))
checked = subprocess.run([sys.executable, '-m', 'pytest', 'test_anchored_mnist_standalone.py', '-q', '-p', 'no:cacheprovider'], env=dict(os.environ, JAX_PLATFORMS='cpu'), capture_output=True, text=True, timeout=180)
print(checked.stdout, checked.stderr)
checked.check_returncode()


In [ ]:
# Public MNIST mirror and checksums are those listed in torchvision/datasets/mnist.py.
from urllib.request import urlretrieve
raw = Path('/content/mnist_raw')
raw.mkdir(exist_ok=True)
resources = {
    'train-images-idx3-ubyte.gz': 'f68b3c2dcbeaaa9fbdd348bbdeb94873',
    'train-labels-idx1-ubyte.gz': 'd53e105ee54ea40749a09fcbcd1e9432',
    't10k-images-idx3-ubyte.gz': '9fb629c4189551a2d022fa330f9573f3',
    't10k-labels-idx1-ubyte.gz': 'ec29112dd5afa0611ce80d1b7f02629c',
}
for name, digest in resources.items():
    path = raw / name
    if not path.exists():
        urlretrieve('https://ossci-datasets.s3.amazonaws.com/mnist/' + name, path)
    assert hashlib.md5(path.read_bytes()).hexdigest() == digest, name
print('All four MNIST files verified')


In [ ]:
import datetime, threading
out = Path('/content') / ('anchored_mnist_' + datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
protocol = 'rate-sweep'
command = [sys.executable, '-u', 'anchored_mnist.py', '--require-gpu', '--protocol', protocol, '--data-dir', str(raw), '--output-dir', str(out), '--steps', '200', '--tune-steps', '50', '--budget-seconds', '1800']
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
timer = threading.Timer(1860, process.kill)
timer.daemon = True
timer.start()
try:
    with Path('/content/mnist_process.log').open('w') as log:
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
    status = process.wait()
finally:
    timer.cancel()
print('Process exit code:', status)
if (out / 'summary.json').exists():
    summary = json.loads((out / 'summary.json').read_text())
    print({k: summary.get(k) for k in ('status', 'solver', 'all_methods_completed', 'elapsed_seconds')})
assert status == 0, 'Inspect saved evidence; a zero exit code alone is not a scientific pass'


In [ ]:
import shutil
shutil.copy('anchored_mnist.py', out / 'source.py')
shutil.copy('/content/mnist_process.log', out / 'process.log')
(out / 'tests.log').write_text(checked.stdout + checked.stderr)
(out / 'environment.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))
evidence = shutil.make_archive(str(out), 'zip', out)
print('Saved evidence:', evidence)
files.download(evidence)
